# GEMS P6 — full GPU benchmark, recovery integration & DrivenData submission | Colab GPU (high-RAM)

One run: **P6 Recovery → P6 Boost A–E → GPU (W1 extra CNN seeds + rot180 TTA) → one-interface benchmark → nested selection → zone-stitched full-resolution submission → validation gate → manual upload → leaderboard log.** Every stage is a repository function (`gems_bench/p6/full_gpu.py`); no modelling code lives in this notebook.

* Reuses (never rewrites) the frozen W2 stacker maps, the M9 B0/B1 maps, the boost A grid, the P6.1 recovery tables and any finished boost B–D / W1 grids from `MyDrive/GEMS_2/outputs/p6_fast_track`.
* Writes everything new under `MyDrive/GEMS_2/outputs/p6_boost/` (`audit/`, `gpu/`, `benchmark/`, `submissions/`, `figures/`, `leaderboard/`, `reports/`), each with `manifest.json` + `status.json`.
* Resumable: after a disconnect run sections 0–3 again; finished stages reload from their fingerprint, W1 seed experiments resume from their checkpoints.
* **Nothing is uploaded.** Every number comes from your Colab run; the authoring session executed nothing. Docs: `docs/p6/P6_FULL_GPU.md`, audit: `docs/p6/P6_RECOVERY_TO_BOOST_AUDIT.md`.

## 0 — Configuration (edit before running)

`FULL_GPU_CONFIG['mode']`: `'full'` needs a GPU runtime (Runtime → Change runtime type → GPU; L4 / A100 when available, T4 works) and stops with `GPURequired` otherwise; `'recovery'` trains nothing and runs on CPU. Keep `M9_CONFIG['profile']` identical to the boost A run (`PILOT`): the benchmark refuses grids whose protocol keys differ. Pin the pushed commit in `CONFIG['ref']` / `EXPECT['commit']`.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                      # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False,
    'train_baselines': [],
    'run_full_inference': False,
    'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': True,
    'seed': 2026,
}
PRIVATE_REPO = False
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
M9_CONFIG = {'profile': 'PILOT', 'overwrite_existing': False}      # same profile as the boost A grid
P6_CONFIG = {'overwrite_existing': False}
BOOST_CONFIG = {'geology': {'backfill_missing': False}}
RECOVERY_CONFIG = {}

FULL_GPU_CONFIG = {
    'mode': 'full',                    # 'full' (GPU seeds + TTA) | 'recovery' (reuse only, CPU)
    # 'gpu': {'fold_average_maps': True},                # diagnostic fold-average B0/B1 maps (never selected)
    # 'selection': {'geo_rule': {'mode': 'gate'}},      # reproduce the P6.1 absolute geo floor 0.1758
}

FULL_RUN = {
    'audit': True,
    'gpu': True,                       # no-op (recorded) in mode 'recovery'
    'g02_backfill': False,             # explicit zone-2 G02 backfill (CPU, ~0.5-1 h); needed by boost B
    'boost_geology': False,            # boost B (S0-S3: G02 / G01 wavelet priors) once every G02 raster exists
    'boost_robustness': False,         # boost C (C1 / C2)
    'boost_hybrid': False,             # boost D (D1 / D2; benchmarked, no full-region builder)
    'benchmark': True,
    'figures': True,
    'submission': True,
    'report': True,
}

# After a MANUAL upload to DrivenData, type the public score here and re-run section 10 only:
LEADERBOARD_ENTRY = None   # {'submission': 'submission_p6_boost.tif', 'public_score': 0.0, 'date': 'YYYY-MM-DD', 'notes': ''}

EXPECT = {
    'notebook_version': 'p6g-nb.1',
    'code_rev': 'p6b.1-r3',
    'recovery_code_rev': 'p6r.1-r1',
    'full_code_rev': 'p6g.1-r1',
    'commit': None,
}

## 1 — Google Drive, strict repository clone, version lock

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
    if not token:
        raise RuntimeError('PRIVATE_REPO=True but GITHUB_TOKEN is missing in Colab Secrets')
    hdr = base64.b64encode(('x-access-token:' + token).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'p6g_colab_active', 'origin/' + ref)
else:                                   # a SHA / tag must resolve; never fall back to main
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: restart the runtime (stale module code)')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('Requested:', ref, '| resolved SHA:', HEAD)

## 2 — Start-up check + GPU detection (fail fast, before any staging)

Mode `full` without CUDA raises `GPURequired` here: expensive models are never moved to the CPU silently.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display
from gems_bench.p6 import gpu_runtime as GR
from gems_bench.p6.full_gpu import load_full_config, startup_check
START = startup_check(EXPECT, HEAD, FULL_GPU_CONFIG)
print(json.dumps(START, indent=2, default=str))
GPU_INFO = GR.detect()
print(json.dumps(GPU_INFO, indent=2, default=str))
DEVICE = GR.require_gpu(GPU_INFO, load_full_config(FULL_GPU_CONFIG)['gpu'])   # GPURequired in mode full
print('Device:', DEVICE)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'nvidia-smi: no GPU visible')

## 3 — Attach the frozen P6 artifacts (no retraining) + runtime record

Stages the M9 context, B0/B1 maps, `stack/` (W2), `boost/`, `recovery/`, `seeds/` and earlier `outputs/p6_boost`. Missing W2 inputs stop here: W2 is not rerun in this notebook.

In [ ]:
from gems_bench.pipeline import Pipeline
from gems_bench.p6 import P6FastTrack

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for step in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(step)                     # validate + stage cached Drive data, never rebuild
m9 = pipe.m9(M9_CONFIG)
m9.load()
p6 = P6FastTrack(m9, P6_CONFIG)
fg = p6.full_gpu(FULL_GPU_CONFIG, RECOVERY_CONFIG, BOOST_CONFIG)
rc, bst = fg.rc, fg.bst
need = fg.stage_inputs()
missing = [k for k in ('stack/candidate.json', 'stack/features/rows.parquet') if not need.get(k)]
if need.get('stack/maps/*.npy', 0) != 6:
    missing.append('stack/maps/*.npy expected 6, got ' + str(need.get('stack/maps/*.npy', 0)))
if missing:
    raise RuntimeError('Frozen W2 artifacts missing: ' + ', '.join(missing) + ' (restore them on Drive; W2 is not rerun)')
RUNTIME = fg.startup()
print('Output root:', fg.root)
print(json.dumps({k: RUNTIME[k] for k in ('mode', 'device', 'gpu_stages', 'seeds', 'precision')}, indent=2, default=str))
display(p6.budget.table())

## 4 — Artifact audit (REUSE / RECOMPUTE / INCOMPATIBLE / UNKNOWN)

In [ ]:
if FULL_RUN['audit']:
    AU = fg.audit()
    display(AU['table'][['artifact', 'class', 'needed_by', 'note']])
    print(json.dumps(AU['decision'], indent=2, default=str))
    if AU['decision']['blocking']:
        print('STOP: artifacts every candidate needs are not reusable:', AU['decision']['blocking'])

## 5 — GPU: W1 extra CNN seeds + rot180 TTA (W1 GPU cap, resumable)

Extra seeds of the committed B0 / B1 experiments (`seeds.families` in `configs/p6/fast_track.yaml`) are trained one at a time while the W1 GPU budget allows; each produces OOF + discovery maps, so the seed ensemble enters the benchmark through the same protocol. TTA re-predicts every honest map with the checkpoint that made it. Precision follows each checkpoint spec (bf16 / fp16 autocast on CUDA, fp32 sigmoid).

In [ ]:
if FULL_RUN['gpu']:
    GPU = fg.gpu()
    display(pd.DataFrame(GPU['steps']))
    sc = GPU.get('seed_candidate') or {}
    print('Seed candidate:', sc.get('name'), sc.get('gate'), sc.get('gate_reasons'), sc.get('disc_dti'))
    display(p6.budget.table())
    display(GR.memory_report())

## 6 — Optional: boost B / C / D (+ explicit G02 zone backfill)

Off by default. Each stage reloads when finished; its grid then joins the benchmark. Boost B (geological / wavelet priors) needs every G02 unit raster: set `g02_backfill=True` first (the audit must say `READY_FOR_BACKFILL`).

In [ ]:
if any(FULL_RUN[k] for k in ('g02_backfill', 'boost_geology', 'boost_robustness', 'boost_hybrid')):
    V = fg.boost_variants(backfill=FULL_RUN['g02_backfill'], geology=FULL_RUN['boost_geology'],
                          robustness=FULL_RUN['boost_robustness'], hybrid=FULL_RUN['boost_hybrid'])
    print(json.dumps(V, indent=2, default=str))

## 7 — Benchmark + model selection (one interface, nested, predeclared)

Every candidate's saved grid rows go through `export_parity.evaluate_policy`: per outer zone ONE cell chosen on the other zones, applied to that zone in both tasks. Grids are pooled only when protocol keys and truth mass agree with boost A per unit. Unexecuted candidates are listed with their status, never with a number. Selection: nested disc_official DTI first; worst zone ≥ 0.153414; geographic DTI not more than 0.010 below the reference; ensembles must beat their best member; a replacement must gain ≥ 0.005 and be no worse on ≥ 2 of 3 zones. META = honest estimate of the choice itself.

In [ ]:
if FULL_RUN['benchmark']:
    BM = fg.benchmark()
    MS = BM['model_selection']
    print('Objective:', MS['objective'], '| decision:', MS['decision'], '| winner:', MS['winner'], '| baseline:', MS['baseline'])
    print('Gain vs baseline:', MS['gain_vs_baseline'], '| slot-worthy:', MS['slot_worthy'], '| reasons:', MS['reasons'])
    meta = MS.get('meta_nested') or {}
    print('META (choose the candidate, nested):', json.dumps({k: meta.get(k) for k in ('status', 'disc_dti', 'disc_worst', 'chosen_per_zone')}, default=str))
    display(BM['candidate_rankings'])
    display(BM['protocol_consistency'].groupby(['source', 'ok']).size().rename('units').reset_index())
    A = BM['candidate_ablations']
    display(A[[c for c in ('ablation', 'task', 'status', 'n_cells', 'mean_delta', 'win_share', 'delta_disc', 'delta_worst', 'delta_geo') if c in A]])

## 8 — Geological and visual diagnostics (PNG + SVG + plot data)

In [ ]:
if FULL_RUN['figures']:
    FIGS = fg.figures()
    print('Figures:', FIGS)
    for n in ('input_layers', 'zone_dti', 'heldout_errors', 'performance_vs_runtime'):
        p = fg.root / 'figures' / (n + '.png')
        if p.exists():
            display(Image(filename=str(p)))

## 9 — GENERATE AND VALIDATE FINAL DRIVENDATA SUBMISSION

Official format checked against `example_submission.tif` at run time: one float32 band, EPSG:32611, 100 m, template transform / shape, values in [0, 1], NaN exactly where the template is NaN. The winner (and the strongest existing baseline, built the same way, for a fair leaderboard comparison) is rebuilt **zone by zone from the held-out maps that were evaluated on that zone**, with rep / frac / blend from one all-zone selection, on the eligible domain (mapped faults masked). Validation = `gems3.submission.validate` + the export audit (CRS, transform, shape, NaN pattern, range, eligibility, area budget, sidecar = recipe, sha256) + zone-stitched coverage and parity. **A failed check leaves the file `NOT_READY`.** Files are never overwritten and never uploaded.

In [ ]:
if FULL_RUN['submission']:
    SUB = fg.submission()
    SM = SUB['submission_manifest']
    print(json.dumps(SUB['official_spec'], indent=2, default=str)[:1500])
    for f in SM['files']:
        local = fg.root / 'submissions' / f['submission']
        print(f['role'].ljust(8), local, '| validation:', f['validation'], '| readiness:', f['readiness'], '| sha256:', f['sha256'])
    V = fg.read_json('submissions', Path(SM['winner_file']).stem + '_validation') or {}
    print('Winner failed checks:', V.get('failed'))
    display(Image(filename=str(fg.root / 'submissions' / 'submission_preview.png')))
    if FULL_RUN['figures']:
        fg.figures()                   # adds the prediction / difference / continuity maps of the new files
    FINAL = fg.root / 'submissions' / SM['winner_file']
    DRIVE = Path(CONFIG['drive_root']) / fg.cfg['output_dir'] / 'submissions' / SM['winner_file']
    print('=' * 100)
    print('FINAL SUBMISSION ARTIFACT:', FINAL)
    print('DRIVE COPY:', DRIVE, '| exists on Drive:', DRIVE.exists())
    print('VALIDATION STATUS:', SM['winner_status'], '| uploaded: False (manual upload only)')
    print('=' * 100)

## 10 — Leaderboard record (after a MANUAL upload) + local vs public comparison

Upload `submission_p6_boost.tif` by hand on the DrivenData submissions page (3 scored submissions per rolling 7 days). Then fill `LEADERBOARD_ENTRY` in section 0 and run this cell. The row is refused if the file no longer hashes to its manifest or already has a score. The public score is never a selection input.

In [ ]:
from gems_bench.p6 import leaderboard as lb
if LEADERBOARD_ENTRY:
    print(json.dumps(lb.record(fg, **LEADERBOARD_ENTRY), indent=2, default=str))
CMP = lb.compare(fg)
print(json.dumps({k: v for k, v in CMP['summary'].items() if k != 'pair_table'}, indent=2, default=str))
if len(CMP['table']):
    display(CMP['table'][['submission_name', 'candidate', 'local', 'public', 'gap_public_minus_local', 'possible_causes']])

## 11 — Report, registry, Drive synchronisation

In [ ]:
if FULL_RUN['report']:
    REP = fg.report()
    display(REP['status'])
    display(Markdown(REP['markdown']))
fg.push()
p6.push()
display(p6.budget.table())
print('Artifacts:', fg.root)

## Execution checklist

1. GPU runtime, `mode='full'`: run 0 → 11 in order. First read `audit/artifact_audit.csv`, then `benchmark/model_selection.json` and `benchmark/candidate_rankings.csv`, then `submissions/submission_manifest.json` + `submissions/<file>_validation.json`.
2. No GPU available: `mode='recovery'` benchmarks and exports from the saved grids only (no seed ensemble).
3. Geological priors: `g02_backfill=True` → re-run 4 (audit) → `boost_geology=True` → re-run 7 and 9.
4. Upload only a `READY_FOR_UPLOAD` file, by hand; then `LEADERBOARD_ENTRY` + section 10. Never tune on the public score.
5. Disconnect: re-run 0 → 3, then the section you were in; finished stages reload (fingerprint + outputs), W1 seed experiments resume.